In [ ]:
import numpy as np

class Node:
    def __init__(self, feature=None, threshold=None, left=None, right=None, prediction=None):
        self.feature = feature
        self.threshold = threshold
        self.left = left
        self.right = right
        self.prediction = prediction
        
class MyDecisionTree:
    def __init__(self, max_depth=5, min_samples_split=2):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.tree = None

    def _gini(self, y):
        if len(y) == 0:
            return 0
        proportions=np.unique(y, return_counts=True)[1]/len(y)
        return 1-np.sum(proportions**2)

    def _split_gain(self, y, left_y, right_y):
        parent_impurity = self._gini(y)
        weighted_children = (
            len(left_y) / len(y) * self._gini(left_y)
            + len(right_y) / len(y) * self._gini(right_y)
        )
        return parent_impurity - weighted_children

    def make_split(self, X, y):
        parent_gini=self._gini(y)
        best_gain=0.0
        best_split=None

        features = X.shape[1]

        for feature in range(features):
            values=np.unique(X[:, feature])
            thresholds=(values[:-1] + values[1:])/2

            for threshold in thresholds:
                left_mask = X[:, feature] <= threshold
                right_mask = X[:, feature] > threshold
                if not left_mask.any() or not right_mask.any():
                    continue

                left_y = y[left_mask]
                right_y = y[right_mask]

                gain=self._split_gain(y,left_y,right_y)

                if gain > best_gain:
                    best_gain = gain
                    best_split = {
                        "feature": feature,
                        "threshold": threshold,
                        "gain": gain,
                        "left_mask": left_mask,
                        "right_mask": right_mask,
                    }

        return best_split
        
    def _make_leaf(self, y):
        classes, counts = np.unique(y, return_counts=True)
        return Node(prediction=classes[np.argmax(counts)])

    def _build_tree(self, X, y, depth=0):
        if (len(np.unique(y)) == 1 or depth >= self.max_depth or len(y) < self.min_samples_split):
            return self._make_leaf(y)

        split = self.make_split(X, y)

        if split is None:
            return self._make_leaf(y)

        left_mask = split["left_mask"]
        right_mask = split["right_mask"]

        return Node(
            feature=split["feature"],
            threshold=split["threshold"],
            left=self._build_tree(X[left_mask], y[left_mask], depth + 1),
            right=self._build_tree(X[right_mask], y[right_mask], depth + 1)
        )
        
    def fit(self, X, y):
        X = np.asarray(X)
        y = np.asarray(y)
        self.root = self._build_tree(X, y)
        return self
